# Celula 1: Incarcarea Seturilor de Date Brute (Data Ingestion)
In aceasta celula se realizeaza importul fisierelor CSV colectate de la dispozitivele
purtabile Fitbit. Setul de date are o granularitate mixta: date zilnice (activitate, somn),
date orare (pasi, calorii) si date la nivel de secundă (ritm cardiac). De asemenea, includem
datele biometrice de jurnalizare a greutatii.
Toate coloanele de tip data/timp sunt convertite din format text (string) in obiecte
native `datetime` pentru a permite analize temporale corecte in etapele urmatoare.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

daily_activity = pd.read_csv('data/dailyActivity_merged.csv')
hourly_steps = pd.read_csv('data/hourlySteps_merged.csv')
hourly_calories = pd.read_csv('data/hourlyCalories_merged.csv')
sleep_day = pd.read_csv('data/sleepDay_merged.csv')
heartrate = pd.read_csv('data/heartrate_seconds_merged.csv')
weight = pd.read_csv('data/weightLogInfo_merged.csv')

daily_activity['ActivityDate'] = pd.to_datetime(daily_activity['ActivityDate'])
hourly_steps['ActivityHour'] = pd.to_datetime(hourly_steps['ActivityHour'])
hourly_calories['ActivityHour'] = pd.to_datetime(hourly_calories['ActivityHour'])
sleep_day['SleepDay'] = pd.to_datetime(sleep_day['SleepDay'])
heartrate['Time'] = pd.to_datetime(heartrate['Time'])
weight['Date'] = pd.to_datetime(weight['Date'])

print(f"Randuri initiale Activitate: {len(daily_activity)}")
print(f"Randuri initiale Puls: {len(heartrate)}")

# Celula 2: Curatarea Avansata a Datelor si Eliminarea Anomaliilor (Data Cleaning)
Datele provenite de la senzori purtabili contin zgomot structural si erori de inregistrare.
În aceasta celula aplicam filtre biologice și logice stricte pentru asigurarea calitatii datelor:
1. **Filtru Activitate:** Eliminam zilele "moarte" in care ceasul a fost uitat pe noptiera
   (0 pasi sau 1440 de minute de sedentarism total intr-o zi).
2. **Filtru Calorii:** Eliminam inregistrarile cu sub 1000 kcal/zi, valoare sub Rata Metabolica
   Bazala (BMR) necesara supravietuirii unui adult, semn ca senzorul nu a fost purtat continuu.
3. **Filtru Puls:** Eliminam erorile optice ale senzorului PPG (valori biologice aberante sub 40 bpm sau peste 200 bpm).
4. **Filtru Somn:** Eliminam duplicatele de sincronizare si sesiunile de somn sub 30 de minute.

In [15]:
daily_activity = daily_activity[(daily_activity['TotalSteps'] > 0) & (daily_activity['SedentaryMinutes'] < 1440)]

daily_activity = daily_activity[daily_activity['Calories'] >= 1000]

heartrate = heartrate[(heartrate['Value'] >= 40) & (heartrate['Value'] <= 200)]

sleep_day = sleep_day.drop_duplicates()

sleep_day = sleep_day[sleep_day['TotalMinutesAsleep'] >= 30]

print(f"\nRanduri dupa curatare Activitate: {len(daily_activity)}")
print(f"Randuri dupa curatare Puls: {len(heartrate)}")


Rânduri după curățare Activitate: 851
Rânduri după curățare Puls: 2483622


# Celula 3: Transformarea Datelor Zilnice si Calcularea KPI-urilor (Feature Engineering - Daily)
Pentru a pregati modelul de Business Intelligence, imbunatatim tabelul zilnic prin tehnici de
imbinare (Left Join) intre activitatea fizica si starea de somn pe baza cheii compuse `[Id, ActivityDate]`.
Extragem metrici derivate (KPIs de Sanatate) care nu existau in formatul brut:
* **MinutesAwakeInBed:** Timpul de insomnie sau latenta (timpul petrecut treaz in pat).
* **SleepEfficiency:** Eficienta somnului (proportia de somn real din totalul timpului petrecut in pat).
* **Time Intelligence:** Extragerea zilei saptamanii si marcarea weekend-urilor pentru analiza comportamentului social.
* **Activity_Level:** Segmentarea categoriala a utilizatorului bazata pe praguri standard de activitate fizica.

In [16]:
sleep_day = sleep_day.rename(columns={'SleepDay': 'ActivityDate'})

df_zilnic_final = pd.merge(daily_activity, sleep_day, on=['Id', 'ActivityDate'], how='left')

df_zilnic_final['MinutesAwakeInBed'] = df_zilnic_final['TotalTimeInBed'] - df_zilnic_final['TotalMinutesAsleep']
df_zilnic_final['SleepEfficiency'] = (df_zilnic_final['TotalMinutesAsleep'] / df_zilnic_final['TotalTimeInBed']) * 100
df_zilnic_final['DayOfWeek'] = df_zilnic_final['ActivityDate'].dt.day_name()
df_zilnic_final['IsWeekend'] = df_zilnic_final['ActivityDate'].dt.dayofweek.isin([5, 6]).astype(int)

def clasifica_activitate(steps):
    if steps < 5000: return 'Sedentar'
    elif steps < 10000: return 'Moderat Activ'
    else: return 'Foarte Activ'
df_zilnic_final['Activity_Level'] = df_zilnic_final['TotalSteps'].apply(clasifica_activitate)

# Celula 4: Agregarea Temporala si Corelarea Datelor (Time-Series Aggregation)
Ritmul cardiac brut este capturat la nivel de secunde, o granularitate mult prea mare pentru dashboard-uri 
macro de BI. In aceasta celula:
1. Rotunjim marcajele temporale ale pulsului la nivel de ora fixa (`dt.floor('h')`).
2. Agregam datele prin calcularea pulsului **Mediu**, **Minim** (indicator apropiat de ritmul cardiac in repaus) 
   si **Maxim** pe fiecare interval orar pentru fiecare individ.
3. Realizam un `Inner Join` intre pasii orari, calorii si pulsul agregat. Acest tip de imbinare ne asigura 
   ca tabelul orar rezultat va fi perfect curat, eliminand complet valorile nule generate de utilizatorii 
   care nu au detinut sau nu au activat senzorul cardiac.
4. Extragem componenta orara pentru analiza tiparelor circadiene (ex. orele de varf pentru activitate).

In [17]:
heartrate['ActivityHour'] = heartrate['Time'].dt.floor('h')

heartrate_hourly = heartrate.groupby(['Id', 'ActivityHour']).agg(
    AverageHeartRate=('Value', 'mean'),
    MinHeartRate=('Value', 'min'),
    MaxHeartRate=('Value', 'max')
).reset_index()

df_orar_intermediar = pd.merge(hourly_steps, hourly_calories, on=['Id', 'ActivityHour'], how='inner')
df_orar_final = pd.merge(df_orar_intermediar, heartrate_hourly, on=['Id', 'ActivityHour'], how='inner')

df_orar_final['HourOfDay'] = df_orar_final['ActivityHour'].dt.hour
df_orar_final['DayOfWeek'] = df_orar_final['ActivityHour'].dt.day_name()

# Celula 5: Crearea Tabelului de Dimensiune - Profil Utilizator (User Dimension Table)
In modelarea de tip Business Intelligence, este esential sa separam datele de tranzactie 
(fapte) de atributele statice ale subiectilor (dimensiuni). 
Aceasta celula creeaza o tabela de profil utilizator bazata pe datele medicale de greutate. 
Calculam media indicelui de masa corporala (`BMI`) pentru fiecare utilizator si aplicam o regula 
medicala standard pentru clasificarea starii ponderale, oferind o axa critica de filtrare a 
datelor in viitorul sistem de raportare.

In [18]:
user_profile = weight.groupby('Id').agg(
    AverageWeightKg=('WeightKg', 'mean'),
    AverageBMI=('BMI', 'mean')
).reset_index()

def clasifica_bmi(bmi):
    if bmi < 18.5: return 'Subponderal'
    elif bmi < 25: return 'Normal'
    elif bmi < 30: return 'Supraponderal'
    else: return 'Obezitate'
user_profile['BMI_Category'] = user_profile['AverageBMI'].apply(clasifica_bmi)

# Celula 6: Exportul Datelor Optimizate pentru Sisteme BI (Data Staging / Export)
Acesta este pasul final al procesului ETL (Extract, Transform, Load). Salvam datele prelucrate 
in fisiere CSV structurate intr-un director dedicat (`date_prelucrate/`). 
Aceste fisiere reprezinta baza curata, sincronizata si lipsita de erori ce va alimenta 
dashboard-urile interactive sau algoritmii avansati de Machine Learning.

In [19]:
df_zilnic_final.to_csv('date_prelucrate/fitbit_analiza_zilnica.csv', index=False)
df_orar_final.to_csv('date_prelucrate/fitbit_analiza_orara.csv', index=False)
user_profile.to_csv('date_prelucrate/fitbit_profil_utilizatori.csv', index=False)

print("\nDatele au fost curatate riguros de anomalii si salvate cu succes!")


Datele au fost curățate riguros de anomalii și salvate cu succes!


True
